# Chapter 7: Convergence Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/Optimization-for-Machine-Learning/blob/main/notebooks/ch07_convergence_analysis.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Measure empirical convergence rates** of optimization algorithms
2. **Estimate Lipschitz constants** from function evaluations
3. **Understand strong convexity** and its effects on convergence
4. **Analyze condition numbers** and their impact on optimization
5. **Compare theoretical vs empirical** convergence behavior
6. **Visualize different convergence rates**: O(1/t) vs O(1/t^2) vs O(e^{-t})

---

## Table of Contents

1. [Setup and Imports](#1-setup)
2. [Convergence Rate Calculator](#2-convergence-rate)
3. [Lipschitz Constant Estimation](#3-lipschitz)
4. [Strong Convexity Effects](#4-strong-convexity)
5. [Condition Number Experiments](#5-condition-number)
6. [Theoretical vs Empirical Convergence](#6-comparison)
7. [Visualizing Convergence Rates](#7-visualization)
8. [Summary and Key Takeaways](#8-summary)

---

## 1. Setup and Imports <a name="1-setup"></a>

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['legend.fontsize'] = 11

# Color palette for consistent styling
COLORS = {
    'gd': '#2E86AB',      # Blue - Gradient Descent
    'agd': '#A23B72',     # Magenta - Accelerated GD
    'newton': '#F18F01',  # Orange - Newton
    'theory': '#C73E1D',  # Red - Theoretical
    'empirical': '#3B7A57'  # Green - Empirical
}

print("Setup complete!")
print(f"NumPy version: {np.__version__}")

## 2. Convergence Rate Calculator <a name="2-convergence-rate"></a>

The convergence rate tells us how fast an algorithm approaches the optimal solution. Common rates include:

- **Sublinear O(1/t)**: Gradient descent on convex functions
- **Sublinear O(1/t^2)**: Accelerated gradient descent (Nesterov)
- **Linear O(rho^t)**: Gradient descent on strongly convex functions
- **Quadratic**: Newton's method near the optimum

We'll build a tool to measure these rates empirically from optimization trajectories.

In [ ]:
class ConvergenceRateCalculator:
    """
    Calculate empirical convergence rates from optimization trajectories.
    
    Supports detection of:
    - Sublinear rates: O(1/t^p) for various p
    - Linear rates: O(rho^t)
    - Superlinear/Quadratic rates
    """
    
    def __init__(self, errors, f_star=None):
        """
        Initialize with error sequence.
        
        Parameters:
        -----------
        errors : array-like
            Sequence of errors (|f(x_t) - f*| or ||x_t - x*||)
        f_star : float, optional
            Optimal value (if not provided, uses final value)
        """
        self.errors = np.array(errors)
        self.errors = np.maximum(self.errors, 1e-16)  # Avoid log(0)
        self.n_iters = len(errors)
        self.iterations = np.arange(1, self.n_iters + 1)
    
    def estimate_sublinear_rate(self, fit_range=None):
        """
        Estimate sublinear rate O(1/t^p).
        
        Returns the exponent p by fitting log(error) ~ -p * log(t)
        """
        if fit_range is None:
            # Use middle portion to avoid initial transient and numerical issues
            start = max(1, self.n_iters // 4)
            end = self.n_iters
        else:
            start, end = fit_range
        
        t = self.iterations[start:end]
        log_errors = np.log(self.errors[start:end])
        log_t = np.log(t)
        
        # Linear regression: log(error) = -p * log(t) + c
        slope, intercept, r_value, _, _ = linregress(log_t, log_errors)
        
        return {
            'rate_type': 'sublinear',
            'exponent_p': -slope,
            'constant': np.exp(intercept),
            'r_squared': r_value**2,
            'formula': f'O(1/t^{-slope:.3f})'
        }
    
    def estimate_linear_rate(self, fit_range=None):
        """
        Estimate linear rate O(rho^t).
        
        Returns the contraction factor rho by fitting log(error) ~ t * log(rho)
        """
        if fit_range is None:
            start = max(1, self.n_iters // 4)
            end = self.n_iters
        else:
            start, end = fit_range
        
        t = self.iterations[start:end]
        log_errors = np.log(self.errors[start:end])
        
        # Linear regression: log(error) = t * log(rho) + c
        slope, intercept, r_value, _, _ = linregress(t, log_errors)
        
        rho = np.exp(slope)
        
        return {
            'rate_type': 'linear',
            'contraction_rho': rho,
            'constant': np.exp(intercept),
            'r_squared': r_value**2,
            'formula': f'O({rho:.4f}^t)'
        }
    
    def detect_rate_type(self):
        """
        Automatically detect whether convergence is sublinear or linear.
        
        Uses R^2 values to determine best fit.
        """
        sublinear = self.estimate_sublinear_rate()
        linear = self.estimate_linear_rate()
        
        # Prefer linear if R^2 is significantly better
        if linear['r_squared'] > sublinear['r_squared'] + 0.05:
            return linear
        elif sublinear['r_squared'] > linear['r_squared'] + 0.05:
            return sublinear
        else:
            # If similar, use the one with better R^2
            return linear if linear['r_squared'] > sublinear['r_squared'] else sublinear
    
    def compute_convergence_ratio(self, k=10):
        """
        Compute local convergence ratios: error[t+k] / error[t]
        
        Useful for detecting quadratic convergence.
        """
        ratios = []
        for i in range(len(self.errors) - k):
            if self.errors[i] > 1e-15:
                ratios.append(self.errors[i + k] / self.errors[i])
        return np.array(ratios)
    
    def summary(self):
        """Generate a complete convergence analysis summary."""
        detected = self.detect_rate_type()
        sublinear = self.estimate_sublinear_rate()
        linear = self.estimate_linear_rate()
        
        print("=" * 60)
        print("CONVERGENCE RATE ANALYSIS")
        print("=" * 60)
        print(f"\nNumber of iterations: {self.n_iters}")
        print(f"Initial error: {self.errors[0]:.6e}")
        print(f"Final error: {self.errors[-1]:.6e}")
        print(f"Total reduction: {self.errors[0] / self.errors[-1]:.2e}x")
        print("\n" + "-" * 40)
        print("SUBLINEAR FIT: O(1/t^p)")
        print(f"  Exponent p = {sublinear['exponent_p']:.4f}")
        print(f"  R^2 = {sublinear['r_squared']:.4f}")
        print("\n" + "-" * 40)
        print("LINEAR FIT: O(rho^t)")
        print(f"  Contraction rho = {linear['contraction_rho']:.6f}")
        print(f"  R^2 = {linear['r_squared']:.4f}")
        print("\n" + "-" * 40)
        print(f"DETECTED RATE: {detected['formula']}")
        print("=" * 60)
        
        return detected

# Test functions for demonstration
print("ConvergenceRateCalculator class defined!")

In [ ]:
# Example: Analyze different convergence behaviors

def run_gradient_descent(f, grad_f, x0, lr, n_iters, f_star=0):
    """Run gradient descent and track errors."""
    x = x0.copy()
    errors = []
    for _ in range(n_iters):
        errors.append(abs(f(x) - f_star))
        x = x - lr * grad_f(x)
    return errors, x

# Test function 1: Quadratic (strongly convex)
# f(x) = 0.5 * x^T A x, where A has eigenvalues in [mu, L]
def create_quadratic(n=10, condition_number=10):
    """Create a quadratic function with specified condition number."""
    # Random orthogonal matrix
    Q, _ = np.linalg.qr(np.random.randn(n, n))
    # Eigenvalues spread between 1 and condition_number
    eigenvalues = np.linspace(1, condition_number, n)
    A = Q @ np.diag(eigenvalues) @ Q.T
    
    def f(x):
        return 0.5 * x @ A @ x
    
    def grad_f(x):
        return A @ x
    
    return f, grad_f, A, 1, condition_number  # mu=1, L=condition_number

# Create test problem
n = 10
f, grad_f, A, mu, L = create_quadratic(n, condition_number=50)
x0 = np.random.randn(n)

# Run GD with optimal learning rate
lr_optimal = 2 / (mu + L)
errors_gd, _ = run_gradient_descent(f, grad_f, x0, lr_optimal, n_iters=200)

# Analyze convergence
calculator = ConvergenceRateCalculator(errors_gd)
result = calculator.summary()

In [ ]:
# Visualize the convergence analysis

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Error vs iterations (log scale)
ax1 = axes[0]
ax1.semilogy(errors_gd, 'b-', linewidth=2, label='GD Error')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Error (log scale)')
ax1.set_title('Convergence of Gradient Descent')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Log-log plot for sublinear rate
ax2 = axes[1]
t = np.arange(1, len(errors_gd) + 1)
ax2.loglog(t, errors_gd, 'b.', markersize=4, alpha=0.5, label='Empirical')
# Fit line
sublinear = calculator.estimate_sublinear_rate()
fitted_sublinear = sublinear['constant'] / t**sublinear['exponent_p']
ax2.loglog(t, fitted_sublinear, 'r--', linewidth=2, 
           label=f'Fit: O(1/t^{sublinear["exponent_p"]:.2f})')
ax2.set_xlabel('Iteration (log scale)')
ax2.set_ylabel('Error (log scale)')
ax2.set_title(f'Sublinear Fit (R^2={sublinear["r_squared"]:.4f})')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Semi-log plot for linear rate
ax3 = axes[2]
ax3.semilogy(t, errors_gd, 'b.', markersize=4, alpha=0.5, label='Empirical')
# Fit line
linear = calculator.estimate_linear_rate()
fitted_linear = linear['constant'] * linear['contraction_rho']**t
ax3.semilogy(t, fitted_linear, 'r--', linewidth=2, 
             label=f'Fit: O({linear["contraction_rho"]:.4f}^t)')
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Error (log scale)')
ax3.set_title(f'Linear Fit (R^2={linear["r_squared"]:.4f})')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('convergence_rate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest fit: {result['formula']}")

## 3. Lipschitz Constant Estimation <a name="3-lipschitz"></a>

The **Lipschitz constant L** of a gradient measures how fast the gradient can change:

$$\|\nabla f(x) - \nabla f(y)\| \leq L \|x - y\|$$

This is crucial because:
- The optimal learning rate for gradient descent is $\alpha = 1/L$
- Convergence guarantees depend on L
- It determines the "smoothness" of the function

We'll estimate L empirically from function/gradient evaluations.

In [ ]:
class LipschitzEstimator:
    """
    Estimate the Lipschitz constant of a gradient from samples.
    
    Methods:
    1. Random sampling: Sample many pairs (x, y) and compute max ratio
    2. Grid-based: Systematic search in a bounded domain
    3. Trajectory-based: Use optimization trajectory points
    """
    
    def __init__(self, grad_f, domain_bounds=None):
        """
        Parameters:
        -----------
        grad_f : callable
            Gradient function
        domain_bounds : tuple, optional
            (lower_bound, upper_bound) for sampling domain
        """
        self.grad_f = grad_f
        self.domain_bounds = domain_bounds or (-10, 10)
    
    def estimate_from_samples(self, n_samples=1000, dim=10):
        """
        Estimate L by sampling random pairs of points.
        
        L = max_{x,y} ||grad_f(x) - grad_f(y)|| / ||x - y||
        """
        max_ratio = 0
        low, high = self.domain_bounds
        
        best_pair = None
        all_ratios = []
        
        for _ in range(n_samples):
            x = np.random.uniform(low, high, dim)
            y = np.random.uniform(low, high, dim)
            
            if np.linalg.norm(x - y) < 1e-10:
                continue
            
            grad_diff = np.linalg.norm(self.grad_f(x) - self.grad_f(y))
            point_diff = np.linalg.norm(x - y)
            
            ratio = grad_diff / point_diff
            all_ratios.append(ratio)
            
            if ratio > max_ratio:
                max_ratio = ratio
                best_pair = (x.copy(), y.copy())
        
        return {
            'L_estimate': max_ratio,
            'mean_ratio': np.mean(all_ratios),
            'std_ratio': np.std(all_ratios),
            'best_pair': best_pair,
            'n_samples': n_samples
        }
    
    def estimate_from_trajectory(self, trajectory):
        """
        Estimate L from an optimization trajectory.
        
        Often gives good estimates as the trajectory visits "interesting" points.
        """
        max_ratio = 0
        trajectory = np.array(trajectory)
        n_points = len(trajectory)
        all_ratios = []
        
        for i in range(n_points):
            for j in range(i + 1, min(i + 50, n_points)):  # Limit to nearby points
                x, y = trajectory[i], trajectory[j]
                
                if np.linalg.norm(x - y) < 1e-10:
                    continue
                
                grad_diff = np.linalg.norm(self.grad_f(x) - self.grad_f(y))
                point_diff = np.linalg.norm(x - y)
                
                ratio = grad_diff / point_diff
                all_ratios.append(ratio)
                
                if ratio > max_ratio:
                    max_ratio = ratio
        
        return {
            'L_estimate': max_ratio,
            'mean_ratio': np.mean(all_ratios) if all_ratios else 0,
            'std_ratio': np.std(all_ratios) if all_ratios else 0,
            'n_pairs': len(all_ratios)
        }
    
    def local_lipschitz(self, x, epsilon=0.1, n_samples=100):
        """
        Estimate local Lipschitz constant around point x.
        """
        dim = len(x)
        max_ratio = 0
        
        for _ in range(n_samples):
            direction = np.random.randn(dim)
            direction = direction / np.linalg.norm(direction)
            y = x + epsilon * direction
            
            grad_diff = np.linalg.norm(self.grad_f(x) - self.grad_f(y))
            point_diff = np.linalg.norm(x - y)
            
            ratio = grad_diff / point_diff
            if ratio > max_ratio:
                max_ratio = ratio
        
        return max_ratio

print("LipschitzEstimator class defined!")

In [ ]:
# Demonstrate Lipschitz estimation on our quadratic function

# For f(x) = 0.5 * x^T A x, the true Lipschitz constant is max eigenvalue of A
true_L = L  # From our earlier quadratic construction

# Create estimator
estimator = LipschitzEstimator(grad_f, domain_bounds=(-5, 5))

# Method 1: Random sampling
result_samples = estimator.estimate_from_samples(n_samples=5000, dim=n)
print("=" * 50)
print("LIPSCHITZ CONSTANT ESTIMATION")
print("=" * 50)
print(f"\nTrue L (max eigenvalue): {true_L:.4f}")
print(f"\nMethod 1: Random Sampling")
print(f"  Estimated L: {result_samples['L_estimate']:.4f}")
print(f"  Mean ratio: {result_samples['mean_ratio']:.4f}")
print(f"  Std ratio: {result_samples['std_ratio']:.4f}")
print(f"  Error: {abs(result_samples['L_estimate'] - true_L)/true_L * 100:.2f}%")

# Method 2: From optimization trajectory
# First, run GD and save trajectory
def run_gd_with_trajectory(f, grad_f, x0, lr, n_iters):
    x = x0.copy()
    trajectory = [x.copy()]
    for _ in range(n_iters):
        x = x - lr * grad_f(x)
        trajectory.append(x.copy())
    return trajectory

trajectory = run_gd_with_trajectory(f, grad_f, x0, lr_optimal, 100)
result_traj = estimator.estimate_from_trajectory(trajectory)

print(f"\nMethod 2: From Trajectory")
print(f"  Estimated L: {result_traj['L_estimate']:.4f}")
print(f"  Error: {abs(result_traj['L_estimate'] - true_L)/true_L * 100:.2f}%")

# Method 3: Local Lipschitz at various points
print(f"\nMethod 3: Local Lipschitz Constants")
test_points = [np.zeros(n), np.ones(n), np.random.randn(n)]
for i, point in enumerate(test_points):
    local_L = estimator.local_lipschitz(point)
    print(f"  Point {i+1}: L_local = {local_L:.4f}")

In [ ]:
# Visualize how Lipschitz estimate converges with more samples

sample_sizes = [50, 100, 200, 500, 1000, 2000, 5000]
estimates = []
errors = []

for n_samples in sample_sizes:
    result = estimator.estimate_from_samples(n_samples=n_samples, dim=n)
    estimates.append(result['L_estimate'])
    errors.append(abs(result['L_estimate'] - true_L) / true_L * 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Estimate vs sample size
ax1 = axes[0]
ax1.semilogx(sample_sizes, estimates, 'bo-', markersize=8, linewidth=2, label='Estimated L')
ax1.axhline(y=true_L, color='r', linestyle='--', linewidth=2, label=f'True L = {true_L}')
ax1.fill_between(sample_sizes, true_L * 0.95, true_L * 1.05, alpha=0.2, color='r', label='5% tolerance')
ax1.set_xlabel('Number of Samples')
ax1.set_ylabel('Lipschitz Estimate')
ax1.set_title('Lipschitz Estimation vs Sample Size')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Relative error
ax2 = axes[1]
ax2.semilogx(sample_sizes, errors, 'go-', markersize=8, linewidth=2)
ax2.set_xlabel('Number of Samples')
ax2.set_ylabel('Relative Error (%)')
ax2.set_title('Estimation Error vs Sample Size')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lipschitz_estimation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Strong Convexity Effects <a name="4-strong-convexity"></a>

A function is **strongly convex** with parameter $\mu > 0$ if:

$$f(y) \geq f(x) + \nabla f(x)^T(y-x) + \frac{\mu}{2}\|y-x\|^2$$

Strong convexity guarantees:
- **Unique global minimum**
- **Linear convergence** for gradient descent: $f(x_t) - f^* \leq \rho^t (f(x_0) - f^*)$
- Contraction factor: $\rho = 1 - \mu/L = (\kappa - 1)/(\kappa + 1)$

where $\kappa = L/\mu$ is the **condition number**.

Let's explore how strong convexity affects convergence.

In [ ]:
def analyze_strong_convexity_effects():
    """
    Compare convergence with different strong convexity parameters.
    """
    np.random.seed(42)
    n = 20
    n_iters = 500
    
    # Different strong convexity parameters (min eigenvalue)
    mu_values = [0.1, 0.5, 1.0, 2.0, 5.0]
    L_fixed = 10.0  # Fixed Lipschitz constant
    
    results = {}
    
    for mu in mu_values:
        # Create quadratic with specific mu and L
        Q, _ = np.linalg.qr(np.random.randn(n, n))
        # Set eigenvalues: min = mu, max = L_fixed
        eigenvalues = np.linspace(mu, L_fixed, n)
        A = Q @ np.diag(eigenvalues) @ Q.T
        
        def f(x):
            return 0.5 * x @ A @ x
        
        def grad_f(x):
            return A @ x
        
        # Optimal learning rate
        lr = 2 / (mu + L_fixed)
        
        # Run GD
        x0 = np.random.randn(n)
        errors, _ = run_gradient_descent(f, grad_f, x0, lr, n_iters)
        
        # Theoretical rate
        kappa = L_fixed / mu
        rho_theory = (kappa - 1) / (kappa + 1)
        
        results[mu] = {
            'errors': errors,
            'kappa': kappa,
            'rho_theory': rho_theory,
            'lr': lr
        }
    
    return results, mu_values

results, mu_values = analyze_strong_convexity_effects()

# Display results
print("=" * 70)
print("STRONG CONVEXITY EFFECTS (L = 10 fixed)")
print("=" * 70)
print(f"{'mu':<8} {'kappa':<10} {'rho_theory':<12} {'Final Error':<15} {'Convergence'}")
print("-" * 70)
for mu in mu_values:
    r = results[mu]
    final_err = r['errors'][-1]
    print(f"{mu:<8.1f} {r['kappa']:<10.1f} {r['rho_theory']:<12.4f} {final_err:<15.2e} {'Linear'}")

In [ ]:
# Visualize strong convexity effects

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Color map for different mu values
colors = plt.cm.viridis(np.linspace(0, 0.9, len(mu_values)))

# Plot 1: Convergence curves
ax1 = axes[0]
for mu, color in zip(mu_values, colors):
    ax1.semilogy(results[mu]['errors'], color=color, linewidth=2, 
                 label=f'mu={mu}, kappa={results[mu]["kappa"]:.1f}')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Error (log scale)')
ax1.set_title('Convergence vs Strong Convexity')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Iterations to epsilon-accuracy
ax2 = axes[1]
epsilon = 1e-6
iters_to_epsilon = []
for mu in mu_values:
    errors = results[mu]['errors']
    try:
        iters = next(i for i, e in enumerate(errors) if e < epsilon)
    except StopIteration:
        iters = len(errors)
    iters_to_epsilon.append(iters)

ax2.bar(range(len(mu_values)), iters_to_epsilon, color=colors)
ax2.set_xticks(range(len(mu_values)))
ax2.set_xticklabels([f'mu={mu}' for mu in mu_values], rotation=45)
ax2.set_ylabel('Iterations to epsilon = 1e-6')
ax2.set_title('Speed vs Strong Convexity')
ax2.grid(True, alpha=0.3, axis='y')

# Plot 3: Theoretical vs empirical contraction
ax3 = axes[2]
rho_theory = [results[mu]['rho_theory'] for mu in mu_values]
kappas = [results[mu]['kappa'] for mu in mu_values]

# Estimate empirical contraction from last 50% of iterations
rho_empirical = []
for mu in mu_values:
    errors = results[mu]['errors']
    mid = len(errors) // 2
    if errors[-1] > 1e-15 and errors[mid] > 1e-15:
        rho = (errors[-1] / errors[mid]) ** (1 / (len(errors) - mid))
    else:
        rho = 0
    rho_empirical.append(rho)

x = np.arange(len(mu_values))
width = 0.35
ax3.bar(x - width/2, rho_theory, width, label='Theoretical', color=COLORS['theory'])
ax3.bar(x + width/2, rho_empirical, width, label='Empirical', color=COLORS['empirical'])
ax3.set_xticks(x)
ax3.set_xticklabels([f'kappa={k:.0f}' for k in kappas], rotation=45)
ax3.set_ylabel('Contraction Factor rho')
ax3.set_title('Theoretical vs Empirical Contraction')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('strong_convexity_effects.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Condition Number Experiments <a name="5-condition-number"></a>

The **condition number** $\kappa = L/\mu$ is perhaps the most important quantity in optimization:

- $\kappa = 1$: Perfect conditioning, fastest convergence
- $\kappa = 10$: Well-conditioned, fast convergence
- $\kappa = 100$: Moderately ill-conditioned
- $\kappa = 1000+$: Ill-conditioned, slow convergence

For gradient descent, the number of iterations to reach accuracy $\epsilon$ is:
$$t = O(\kappa \log(1/\epsilon))$$

For accelerated methods (Nesterov), it's:
$$t = O(\sqrt{\kappa} \log(1/\epsilon))$$

This $\sqrt{\kappa}$ vs $\kappa$ difference is huge for ill-conditioned problems!

In [ ]:
def nesterov_accelerated_gd(f, grad_f, x0, L, n_iters, f_star=0):
    """
    Nesterov's Accelerated Gradient Descent.
    
    Updates:
        y_{t+1} = x_t - (1/L) * grad_f(x_t)
        x_{t+1} = y_{t+1} + beta_t * (y_{t+1} - y_t)
    
    where beta_t = (t-1)/(t+2) for convex functions.
    """
    x = x0.copy()
    y = x0.copy()
    y_old = x0.copy()
    errors = []
    
    for t in range(1, n_iters + 1):
        errors.append(abs(f(x) - f_star))
        
        # Gradient step
        y_new = x - (1/L) * grad_f(x)
        
        # Momentum
        beta = (t - 1) / (t + 2)
        x = y_new + beta * (y_new - y_old)
        
        y_old = y_new
    
    return errors, x


def condition_number_experiment():
    """
    Compare GD vs Accelerated GD across different condition numbers.
    """
    np.random.seed(42)
    n = 50
    n_iters = 1000
    
    kappas = [2, 5, 10, 25, 50, 100, 200]
    
    results = {'gd': {}, 'agd': {}}
    
    for kappa in kappas:
        # Create problem with condition number kappa
        Q, _ = np.linalg.qr(np.random.randn(n, n))
        mu, L = 1.0, kappa
        eigenvalues = np.linspace(mu, L, n)
        A = Q @ np.diag(eigenvalues) @ Q.T
        
        def f(x):
            return 0.5 * x @ A @ x
        
        def grad_f(x):
            return A @ x
        
        x0 = np.random.randn(n)
        
        # Gradient Descent
        lr_gd = 2 / (mu + L)
        errors_gd, _ = run_gradient_descent(f, grad_f, x0, lr_gd, n_iters)
        
        # Accelerated GD
        errors_agd, _ = nesterov_accelerated_gd(f, grad_f, x0, L, n_iters)
        
        results['gd'][kappa] = errors_gd
        results['agd'][kappa] = errors_agd
    
    return results, kappas

results_kappa, kappas = condition_number_experiment()

# Summary table
print("=" * 80)
print("CONDITION NUMBER EXPERIMENT: GD vs Accelerated GD")
print("=" * 80)
print(f"{'kappa':<10} {'GD iters to 1e-6':<20} {'AGD iters to 1e-6':<20} {'Speedup':<10}")
print("-" * 80)

epsilon = 1e-6
for kappa in kappas:
    # Find iterations to epsilon
    try:
        iters_gd = next(i for i, e in enumerate(results_kappa['gd'][kappa]) if e < epsilon)
    except StopIteration:
        iters_gd = float('inf')
    
    try:
        iters_agd = next(i for i, e in enumerate(results_kappa['agd'][kappa]) if e < epsilon)
    except StopIteration:
        iters_agd = float('inf')
    
    if iters_agd > 0 and iters_agd != float('inf'):
        speedup = iters_gd / iters_agd if iters_gd != float('inf') else float('inf')
        print(f"{kappa:<10} {iters_gd:<20} {iters_agd:<20} {speedup:<10.2f}x")
    else:
        print(f"{kappa:<10} {iters_gd:<20} {iters_agd:<20} {'N/A':<10}")

In [ ]:
# Visualize condition number effects

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Select a few kappas for clear visualization
selected_kappas = [5, 25, 100, 200]

# Plot 1: GD convergence for different kappas
ax1 = axes[0, 0]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(selected_kappas)))
for kappa, color in zip(selected_kappas, colors):
    ax1.semilogy(results_kappa['gd'][kappa][:500], color=color, linewidth=2, 
                 label=f'kappa={kappa}')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Error (log scale)')
ax1.set_title('Gradient Descent: Effect of Condition Number')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: AGD convergence for different kappas
ax2 = axes[0, 1]
for kappa, color in zip(selected_kappas, colors):
    ax2.semilogy(results_kappa['agd'][kappa][:500], color=color, linewidth=2, 
                 label=f'kappa={kappa}')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Error (log scale)')
ax2.set_title('Accelerated GD: Effect of Condition Number')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: GD vs AGD for kappa=100
ax3 = axes[1, 0]
kappa_compare = 100
ax3.semilogy(results_kappa['gd'][kappa_compare], color=COLORS['gd'], 
             linewidth=2, label='Gradient Descent')
ax3.semilogy(results_kappa['agd'][kappa_compare], color=COLORS['agd'], 
             linewidth=2, label='Accelerated GD')
ax3.set_xlabel('Iteration')
ax3.set_ylabel('Error (log scale)')
ax3.set_title(f'GD vs AGD (kappa={kappa_compare})')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Iterations to epsilon vs kappa
ax4 = axes[1, 1]
iters_gd_list = []
iters_agd_list = []
epsilon = 1e-6

for kappa in kappas:
    try:
        iters_gd = next(i for i, e in enumerate(results_kappa['gd'][kappa]) if e < epsilon)
    except StopIteration:
        iters_gd = 1000
    
    try:
        iters_agd = next(i for i, e in enumerate(results_kappa['agd'][kappa]) if e < epsilon)
    except StopIteration:
        iters_agd = 1000
    
    iters_gd_list.append(iters_gd)
    iters_agd_list.append(iters_agd)

ax4.loglog(kappas, iters_gd_list, 'o-', color=COLORS['gd'], linewidth=2, 
           markersize=8, label='GD: O(kappa)')
ax4.loglog(kappas, iters_agd_list, 's-', color=COLORS['agd'], linewidth=2, 
           markersize=8, label='AGD: O(sqrt(kappa))')

# Theoretical lines
kappa_theory = np.array(kappas)
ax4.loglog(kappa_theory, 5 * kappa_theory, '--', color=COLORS['gd'], alpha=0.5, 
           label='Theory: O(kappa)')
ax4.loglog(kappa_theory, 15 * np.sqrt(kappa_theory), '--', color=COLORS['agd'], 
           alpha=0.5, label='Theory: O(sqrt(kappa))')

ax4.set_xlabel('Condition Number kappa')
ax4.set_ylabel('Iterations to epsilon=1e-6')
ax4.set_title('Iteration Complexity vs Condition Number')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('condition_number_effects.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Theoretical vs Empirical Convergence <a name="6-comparison"></a>

Let's compare theoretical convergence bounds with empirical behavior:

**Gradient Descent on strongly convex functions:**
- Theory: $f(x_t) - f^* \leq \left(\frac{\kappa - 1}{\kappa + 1}\right)^t (f(x_0) - f^*)$

**Accelerated Gradient Descent:**
- Theory: $f(x_t) - f^* \leq 2L\|x_0 - x^*\|^2 \left(\frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1}\right)^{2t}$

Theoretical bounds are often pessimistic - let's see by how much!

In [ ]:
def theoretical_vs_empirical_comparison():
    """
    Compare theoretical convergence bounds with empirical performance.
    """
    np.random.seed(42)
    n = 30
    n_iters = 300
    
    # Create a well-defined problem
    kappa = 50
    mu, L = 1.0, kappa
    Q, _ = np.linalg.qr(np.random.randn(n, n))
    eigenvalues = np.linspace(mu, L, n)
    A = Q @ np.diag(eigenvalues) @ Q.T
    
    def f(x):
        return 0.5 * x @ A @ x
    
    def grad_f(x):
        return A @ x
    
    x0 = np.random.randn(n)
    x_star = np.zeros(n)
    f_star = 0
    
    # Initial values for bounds
    f0 = f(x0)
    dist0 = np.linalg.norm(x0 - x_star)
    
    # Run algorithms
    lr_gd = 2 / (mu + L)
    errors_gd, _ = run_gradient_descent(f, grad_f, x0, lr_gd, n_iters)
    errors_agd, _ = nesterov_accelerated_gd(f, grad_f, x0, L, n_iters)
    
    # Theoretical bounds
    t = np.arange(n_iters)
    
    # GD bound
    rho_gd = (kappa - 1) / (kappa + 1)
    bound_gd = f0 * (rho_gd ** t)
    
    # AGD bound (using simplified form)
    rho_agd = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)) ** 2
    bound_agd = 2 * L * dist0**2 * (rho_agd ** t)
    
    return {
        'empirical_gd': errors_gd,
        'empirical_agd': errors_agd,
        'bound_gd': bound_gd,
        'bound_agd': bound_agd,
        'kappa': kappa,
        'f0': f0,
        'rho_gd': rho_gd,
        'rho_agd': rho_agd
    }

comparison = theoretical_vs_empirical_comparison()

print("=" * 60)
print("THEORETICAL VS EMPIRICAL COMPARISON")
print("=" * 60)
print(f"\nCondition number: {comparison['kappa']}")
print(f"Initial error: {comparison['f0']:.4f}")
print(f"\nGD contraction (theory): {comparison['rho_gd']:.4f}")
print(f"AGD contraction (theory): {comparison['rho_agd']:.4f}")
print(f"\nFinal error GD (empirical): {comparison['empirical_gd'][-1]:.2e}")
print(f"Final error AGD (empirical): {comparison['empirical_agd'][-1]:.2e}")
print(f"Final bound GD (theory): {comparison['bound_gd'][-1]:.2e}")
print(f"Final bound AGD (theory): {comparison['bound_agd'][-1]:.2e}")

In [ ]:
# Visualize theoretical vs empirical

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: GD - Theory vs Empirical
ax1 = axes[0]
ax1.semilogy(comparison['empirical_gd'], color=COLORS['empirical'], linewidth=2.5, 
             label='Empirical GD')
ax1.semilogy(comparison['bound_gd'], '--', color=COLORS['theory'], linewidth=2.5, 
             label='Theoretical Bound')
ax1.fill_between(range(len(comparison['bound_gd'])), 
                 comparison['empirical_gd'], comparison['bound_gd'],
                 alpha=0.2, color='gray', label='Gap (pessimism)')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Error (log scale)')
ax1.set_title(f'Gradient Descent (kappa={comparison["kappa"]})')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim([1e-15, comparison['f0'] * 10])

# Plot 2: AGD - Theory vs Empirical
ax2 = axes[1]
ax2.semilogy(comparison['empirical_agd'], color=COLORS['empirical'], linewidth=2.5, 
             label='Empirical AGD')
ax2.semilogy(comparison['bound_agd'], '--', color=COLORS['theory'], linewidth=2.5, 
             label='Theoretical Bound')
ax2.fill_between(range(len(comparison['bound_agd'])), 
                 np.minimum(comparison['empirical_agd'], comparison['bound_agd']), 
                 comparison['bound_agd'],
                 alpha=0.2, color='gray', label='Gap (pessimism)')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Error (log scale)')
ax2.set_title(f'Accelerated GD (kappa={comparison["kappa"]})')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([1e-15, max(comparison['bound_agd'][0], comparison['f0']) * 10])

plt.tight_layout()
plt.savefig('theoretical_vs_empirical.png', dpi=150, bbox_inches='tight')
plt.show()

# Compute tightness ratio
print("\nTightness Analysis:")
print("-" * 40)
gd_tightness = comparison['bound_gd'] / np.maximum(comparison['empirical_gd'], 1e-16)
agd_tightness = comparison['bound_agd'] / np.maximum(comparison['empirical_agd'], 1e-16)
print(f"GD bound looseness (median): {np.median(gd_tightness[10:100]):.1f}x")
print(f"AGD bound looseness (median): {np.median(agd_tightness[10:100]):.1f}x")

## 7. Visualizing Convergence Rates <a name="7-visualization"></a>

Different convergence rates have dramatically different behavior:

| Rate Type | Formula | Example |
|-----------|---------|--------|
| Sublinear | O(1/t) | GD on convex (not strongly convex) |
| Sublinear | O(1/t^2) | Accelerated GD on convex |
| Linear | O(rho^t) | GD on strongly convex |
| Superlinear | o(rho^t) | Quasi-Newton near optimum |
| Quadratic | O(error^2) | Newton's method near optimum |

Let's visualize these rates to build intuition.

In [ ]:
def visualize_convergence_rates():
    """
    Create comprehensive visualization of different convergence rates.
    """
    t = np.arange(1, 201)
    
    # Different rates (normalized to start at 1)
    rates = {
        'O(1/t) - Sublinear slow': 1 / t,
        'O(1/t^2) - Sublinear fast': 1 / t**2,
        'O(0.95^t) - Linear slow': 0.95**t,
        'O(0.8^t) - Linear fast': 0.8**t,
        'O(0.5^t) - Linear very fast': 0.5**t,
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    colors = plt.cm.tab10(np.linspace(0, 0.5, len(rates)))
    
    # Plot 1: Linear scale
    ax1 = axes[0, 0]
    for (name, values), color in zip(rates.items(), colors):
        ax1.plot(t[:50], values[:50], linewidth=2, color=color, label=name)
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Error')
    ax1.set_title('Convergence Rates (Linear Scale, First 50 iters)')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Log scale
    ax2 = axes[0, 1]
    for (name, values), color in zip(rates.items(), colors):
        ax2.semilogy(t, values, linewidth=2, color=color, label=name)
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Error (log scale)')
    ax2.set_title('Convergence Rates (Log Scale)')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([1e-20, 10])
    
    # Plot 3: Log-log scale (reveals polynomial rates)
    ax3 = axes[1, 0]
    for (name, values), color in zip(rates.items(), colors):
        ax3.loglog(t, values, linewidth=2, color=color, label=name)
    ax3.set_xlabel('Iteration (log scale)')
    ax3.set_ylabel('Error (log scale)')
    ax3.set_title('Log-Log Plot (Lines = Polynomial Rates)')
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Iterations to reach different accuracies
    ax4 = axes[1, 1]
    epsilons = [1e-2, 1e-4, 1e-6, 1e-8, 1e-10]
    
    rate_names_short = ['1/t', '1/t^2', '0.95^t', '0.8^t', '0.5^t']
    x_pos = np.arange(len(epsilons))
    width = 0.15
    
    for i, ((name, values), short_name) in enumerate(zip(rates.items(), rate_names_short)):
        iters_needed = []
        for eps in epsilons:
            try:
                iter_num = next(j for j, v in enumerate(values) if v < eps)
            except StopIteration:
                iter_num = 200  # Max iterations
            iters_needed.append(iter_num)
        ax4.bar(x_pos + i*width - 2*width, iters_needed, width, label=short_name)
    
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels([f'1e-{int(-np.log10(e))}' for e in epsilons])
    ax4.set_xlabel('Target Accuracy')
    ax4.set_ylabel('Iterations Needed')
    ax4.set_title('Iterations to Reach Accuracy')
    ax4.legend(fontsize=9)
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('convergence_rates_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_convergence_rates()

In [ ]:
# Interactive: Crossover points between rates

def find_crossover(rate1, rate2, t_max=1000):
    """Find where two rates cross."""
    t = np.arange(1, t_max + 1)
    vals1 = rate1(t)
    vals2 = rate2(t)
    
    crossings = np.where(np.diff(np.sign(vals1 - vals2)))[0]
    if len(crossings) > 0:
        return crossings[0] + 1
    return None

# Compare O(1/t^2) vs O(0.95^t)
t = np.arange(1, 200)
sublinear = 10 / t**2  # O(1/t^2) with constant 10
linear = 0.95**t       # O(0.95^t)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(t, sublinear, linewidth=2.5, color=COLORS['agd'], label='O(1/t^2) - Accelerated GD')
ax.semilogy(t, linear, linewidth=2.5, color=COLORS['gd'], label='O(0.95^t) - GD on strongly convex')

# Find and mark crossover
cross_idx = np.argmin(np.abs(sublinear - linear))
ax.axvline(x=cross_idx, color='gray', linestyle='--', alpha=0.7)
ax.scatter([cross_idx], [sublinear[cross_idx]], s=100, c='red', zorder=5)
ax.annotate(f'Crossover at t={cross_idx}', xy=(cross_idx, sublinear[cross_idx]),
            xytext=(cross_idx + 20, sublinear[cross_idx] * 10),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=12, color='red')

ax.set_xlabel('Iteration')
ax.set_ylabel('Error (log scale)')
ax.set_title('Rate Comparison: Sublinear vs Linear Convergence')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('rate_crossover.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nKey Insight:")
print(f"The sublinear O(1/t^2) rate starts faster but eventually loses to linear rates.")
print(f"For high accuracy, linear convergence (strongly convex) always wins!")

## 8. Summary and Key Takeaways <a name="8-summary"></a>

### What We Learned

1. **Convergence Rate Analysis**
   - We can measure empirical rates by fitting error curves
   - Sublinear rates show straight lines on log-log plots
   - Linear rates show straight lines on semi-log plots

2. **Lipschitz Constant**
   - L determines the maximum step size: alpha <= 1/L
   - Can be estimated from random sampling or trajectory
   - Local Lipschitz may vary across the domain

3. **Strong Convexity**
   - Guarantees linear convergence for gradient descent
   - Larger mu = faster convergence
   - Creates a "quadratic lower bound" on the function

4. **Condition Number (kappa = L/mu)**
   - Most important quantity for optimization speed
   - GD: O(kappa * log(1/eps)) iterations
   - AGD: O(sqrt(kappa) * log(1/eps)) iterations
   - High kappa = ill-conditioned = slow convergence

5. **Theory vs Practice**
   - Theoretical bounds are often pessimistic
   - Empirical performance usually better than guaranteed
   - But theory gives worst-case guarantees

### Key Formulas

| Algorithm | Rate | Iterations to epsilon |
|-----------|------|----------------------|
| GD (convex) | O(1/t) | O(1/eps) |
| AGD (convex) | O(1/t^2) | O(1/sqrt(eps)) |
| GD (strongly convex) | O(rho^t) | O(kappa * log(1/eps)) |
| AGD (strongly convex) | O(rho^{2t}) | O(sqrt(kappa) * log(1/eps)) |

### Practical Recommendations

1. **Always estimate L** before setting the learning rate
2. **Add regularization** to improve conditioning (increase mu)
3. **Use acceleration** when kappa is large
4. **Monitor convergence** to detect rate type
5. **Theory bounds** are useful but not tight

---

## Next Steps

- Explore adaptive methods (AdaGrad, Adam) that estimate L automatically
- Study preconditioning techniques to reduce condition number
- Investigate stochastic convergence rates

In [ ]:
# Final summary visualization

fig, ax = plt.subplots(figsize=(12, 7))

# Create a summary chart
algorithms = ['GD\n(convex)', 'AGD\n(convex)', 'GD\n(str. convex)', 'AGD\n(str. convex)']
complexity_log_eps = [1000, 32, 50, 7]  # Approximate iterations for kappa=50, eps=1e-6

colors_summary = [COLORS['gd'], COLORS['agd'], COLORS['gd'], COLORS['agd']]
patterns = ['', '', '//', '//']

bars = ax.bar(algorithms, complexity_log_eps, color=colors_summary, edgecolor='black', linewidth=2)

# Add hatching for strongly convex
for bar, pattern in zip(bars, patterns):
    bar.set_hatch(pattern)

ax.set_ylabel('Iterations to epsilon = 1e-6 (approx.)')
ax.set_title('Algorithm Comparison: Iteration Complexity (kappa = 50)')
ax.grid(True, alpha=0.3, axis='y')

# Add annotations
for bar, val in zip(bars, complexity_log_eps):
    ax.annotate(f'{val}', xy=(bar.get_x() + bar.get_width()/2, val),
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=12, fontweight='bold')

# Add legend for hatching
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='gray', edgecolor='black', label='Convex'),
                   Patch(facecolor='gray', edgecolor='black', hatch='//', label='Strongly Convex')]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.savefig('convergence_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("NOTEBOOK COMPLETE!")
print("="*60)
print("\nGenerated figures saved as PNG files in current directory.")
print("Review the Summary section for key takeaways.")